<a href="https://colab.research.google.com/github/Tharun-C0/URBAN-SOUND-CLASSIFICATION-CNN-RANDOM-FOREST/blob/main/final_Urban_Sound_Classification_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install kaggle librosa resampy soundfile
!apt-get install ffmpeg -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 45 not upgraded.


In [ ]:
from google.colab import files
files.upload()  # Upload kaggle.json here

Saving kaggle.json to kaggle (2).json


{'kaggle (2).json': b'{"username":"tharunc2006","key":"6e7e89af201cd695808be84b770e4130"}'}

In [ ]:
import os

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d chrisfilo/urbansound8k --force

Dataset URL: https://www.kaggle.com/datasets/chrisfilo/urbansound8k
License(s): Attribution-NonCommercial 4.0 International (CC BY-NC 4.0)
100% 5.61G/5.61G [05:51<00:00, 17.1MB/s]



In [ ]:
import numpy as np
import pandas as pd
import librosa
import os
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

In [ ]:
!unzip -o -q urbansound8k.zip -d /content/

In [ ]:
import os
print(os.listdir("/content"))

['.config', 'kaggle (2).json', 'fold9', 'fold2', 'fold10', 'fold6', 'UrbanSound8K.csv', 'fold5', 'fold1', 'fold7', 'fold4', 'fold8', 'kaggle.json', 'fold3', 'kaggle (1).json', 'urbansound8k.zip', 'sample_data']


In [ ]:
DATASET_PATH = "/content"

In [ ]:
metadata = pd.read_csv(DATASET_PATH + "/UrbanSound8K.csv")
print(metadata.head())

      slice_file_name    fsID  start        end  salience  fold  classID  \
0    100032-3-0-0.wav  100032    0.0   0.317551         1     5        3   
1  100263-2-0-117.wav  100263   58.5  62.500000         1     5        2   
2  100263-2-0-121.wav  100263   60.5  64.500000         1     5        2   
3  100263-2-0-126.wav  100263   63.0  67.000000         1     5        2   
4  100263-2-0-137.wav  100263   68.5  72.500000         1     5        2   

              class  
0          dog_bark  
1  children_playing  
2  children_playing  
3  children_playing  
4  children_playing  


In [ ]:
def extract_features(file_path):
    try:
        audio, sr = librosa.load(file_path, res_type='kaiser_fast')

        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)

        # Fix length
        if mfcc.shape[1] < 130:
            pad = 130 - mfcc.shape[1]
            mfcc = np.pad(mfcc, ((0,0),(0,pad)), mode='constant')
        else:
            mfcc = mfcc[:, :130]

        return mfcc

    except:
        return None

In [ ]:
features = []
labels = []

for index, row in tqdm(metadata.iterrows(), total=len(metadata)):
    file_path = os.path.join(
        DATASET_PATH,
        "fold" + str(row["fold"]),
        row["slice_file_name"]
    )

    data = extract_features(file_path)

    if data is not None:
        features.append(data)
        labels.append(row["class"])

 41%|████      | 3554/8732 [05:37<06:44, 12.79it/s]/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1323
  warnings.warn(
 95%|█████████▌| 8326/8732 [12:27<00:19, 20.76it/s]/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1103
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1523
  warnings.warn(
100%|██████████| 8732/8732 [12:58<00:00, 11.21it/s]


In [ ]:
X = np.array(features)
y = np.array(labels)

# Normalize
X = X / (np.max(X) + 1e-6)

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)
y = to_categorical(y_encoded)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Reshape
X_train = X_train.reshape(X_train.shape[0], 40, 130, 1)
X_test = X_test.reshape(X_test.shape[0], 40, 130, 1)

In [ ]:
model = Sequential()

model.add(Conv2D(32, (3,3), activation='relu', input_shape=(40,130,1)))
model.add(MaxPooling2D((2,2)))

model.add(Conv2D(64, (3,3), activation='relu'))
model.add(MaxPooling2D((2,2)))

model.add(Conv2D(128, (3,3), activation='relu'))
model.add(MaxPooling2D((2,2)))

model.add(Flatten())

model.add(Dense(256, activation='relu'))
model.add(Dropout(0.4))

model.add(Dense(y.shape[1], activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 17s 32ms/step - accuracy: 0.4269 - loss: 1.5893 - val_accuracy: 0.6033 - val_loss: 1.1599
Epoch 2/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.6311 - loss: 1.0881 - val_accuracy: 0.6554 - val_loss: 0.9441
Epoch 3/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.7035 - loss: 0.8508 - val_accuracy: 0.7413 - val_loss: 0.7548
Epoch 4/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.7613 - loss: 0.7001 - val_accuracy: 0.8163 - val_loss: 0.5775
Epoch 5/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8160 - loss: 0.5509 - val_accuracy: 0.8340 - val_loss: 0.5229
Epoch 6/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8366 - loss: 0.4729 - val_accuracy: 0.8294 - val_loss: 0.5167
Epoch 7/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8736 - loss: 0.3694 - val_accuracy: 0.8512 - val_loss: 0.4423
Epoch 8/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.8949 - loss: 0.3075 - val_accuracy: 

In [ ]:
loss, acc = model.evaluate(X_test, y_test)
print("Accuracy:", acc)

55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9004 - loss: 0.4983
Accuracy: 0.900400698184967


In [ ]:
def predict_sound(file_path):
    feature = extract_features(file_path)

    if feature is None:
        return "Error processing audio"

    feature = feature / (np.max(feature) + 1e-6)
    feature = feature.reshape(1, 40, 130, 1)

    prediction = model.predict(feature)
    return le.inverse_transform([np.argmax(prediction)])[0]

In [ ]:
test_file = "/content/fold1/101415-3-0-2.wav"
print("Prediction:", predict_sound(test_file))

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 910ms/step
Prediction: dog_bark


In [ ]:
from google.colab import files
uploaded = files.upload()

for file in uploaded.keys():
    print("Prediction:", predict_sound(file))

Saving 9031-3-2-0.wav to 9031-3-2-0.wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
Prediction: dog_bark
